#### **Случайная величина имеет распределение Парето**
**p(x) = (θ-1)/x^θ { [1, +inf] }, θ>1**

In [1]:
import numpy as np

### **Выборка объема n**

In [2]:
n = 100
theta = 5
a = theta - 1
m = 1
np.random.seed(67)

x_n = (np.random.pareto(a, n) + 1) * m

print(*x_n, sep='\n')

1.2181492782394967
1.6314916838488003
1.3357662104762218
1.1059603598097307
1.0155885536766278
1.129814224748835
1.0617615480522657
1.9619356283795895
1.3781892594777332
1.0120800122140858
1.5380677070056032
1.2511236596746136
1.0753132088972528
1.0992323812099127
1.0533903526037667
1.1138458987611835
1.0771800071308182
1.0027979917350813
1.1408046948490644
1.1420961377485463
1.0110984255391933
2.171291288633526
1.1351223884754678
1.2075264419940683
1.0685003748672148
1.0966509872155203
1.2222924408845848
1.3609158848206244
2.1525871457696475
1.2589506937383454
1.1372878224938454
1.115830741984349
1.5230607979591424
1.0923463072576878
1.2173211251717622
2.28134566638006
1.0519547392393394
1.38470324995875
1.391457189935009
1.0118795950738406
2.3946002324198914
1.1226866975009127
1.3591344904982867
1.083316808746229
1.0742281555610615
1.0952152543853848
1.0380793435932925
2.517763617215899
1.0021141025954428
1.7545181614214274
1.1155169986945508
1.924074362339163
1.1364503577780187
1.17

### **е) Вычислить полученные доверительные интервалы для доверительной вероятности β=0.95**

In [3]:
β = 0.95
u_0_025 = -1.96
u_0_975 = 1.96

theta_est_p = 1 + n/(np.sum(np.log(x_n)))
median_est = 2**(1/(theta_est_p-1))


# Асимптотический доверительный интервал медианы
start1 = median_est * (1 - (np.log(2)*u_0_975)/(np.sqrt(n)*(theta_est_p-1)))
end1 = median_est * (1 - (np.log(2)*u_0_025)/(np.sqrt(n)*(theta_est_p-1)))

l1 = np.round(end1-start1, 3)
print("Асимптотический доверительный интервал медианы: ")
print(f"{np.round(start1,3)} < xm < {np.round(end1,3)}", end="\n\n")
print("Длина асимптотического доверительного интервала медианы: ")
print(f"l1 = {l1}")

Асимптотический доверительный интервал медианы: 
1.141 < xm < 1.217

Длина асимптотического доверительного интервала медианы: 
l1 = 0.076


In [4]:
# Асимптотический доверительный интервал
start2 = theta_est_p - ((theta_est_p - 1) * u_0_975)/(np.sqrt(n))
end2 = theta_est_p - ((theta_est_p - 1) * u_0_025)/(np.sqrt(n))
l2 = np.round(end2-start2, 3)
print("Асимптотический доверительный интервал: ")
print(f"{np.round(start2,3)} < θ < {np.round(end2,3)}", end="\n\n")
print("Длина асимптотического доверительного интервала: ")
print(f"l2 = {l2}")

Асимптотический доверительный интервал: 
4.379 < θ < 6.027

Длина асимптотического доверительного интервала: 
l2 = 1.648


### **ё) Численно построить бутстраповский доверительный интервал двумя способами β=0.95**

In [5]:
N = 1_000  

def my_function(selection: np.array):
    theta_est_est_p = 1 + n/(np.sum(np.log(selection)))
    return theta_est_p - theta_est_est_p


def non_param_bootstrap(selection: np.array, statistic: function, N: int):
    stat_selection = []
    selection_new = np.random.choice(selection, size=len(selection)*N)

    for i in range(0, len(selection)*N, len(selection)):
        stat_selection.append(statistic(selection_new[i:i+len(selection)]))

    return stat_selection

deltas = non_param_bootstrap(x_n, my_function, N)
deltas = np.sort(deltas)
k1 = int((1-β)/2 * N)
k2 = int((1+β)/2 * N)
start3 = theta_est_p - deltas[k2]
end3 = theta_est_p - deltas[k1]
l3 = np.round(end3-start3, 3)
print("Бутстраповский непараметрический доверительный интервал: ")
print(f"{np.round(start3,3)} < θ < {np.round(end3,3)}", end="\n\n")
print("Длина бутстраповского непараметрического доверительного интервала: ")
print(f"l3 = {l3}")

Бутстраповский непараметрический доверительный интервал: 
4.427 < θ < 6.224

Длина бутстраповского непараметрического доверительного интервала: 
l3 = 1.797


In [6]:
N = 50_000  

def my_model(params: np.array, N:int):
    my_theta = params[0]
    a = my_theta - 1
    m = 1

    selection = (np.random.pareto(a, n*N) + 1) * m

    return selection

def my_params(selection: np.array):
    return np.array([1 + n/(np.sum(np.log(selection)))])

def param_bootstrap(selection: np.array, statistic: function, model: function, params: function, N: int):
    stat_selection = []
    params_ = params(selection)
    selection_new = my_model(params_, N)

    for i in range(0, len(selection)*N, len(selection)):
        stat_selection.append(statistic(selection_new[i:i+len(selection)]))

    return stat_selection

deltas = param_bootstrap(x_n, my_function, my_model, my_params, N)
deltas.sort()
k1 = int((1-β)/2 * N)
k2 = int((1+β)/2 * N)
start4 = theta_est_p - deltas[k2]
end4 = theta_est_p - deltas[k1]
l4 = np.round(end4-start4, 3)
print("Бутстраповский параметрический доверительный интервал: ")
print(f"{np.round(start4,3)} < θ < {np.round(end4,3)}", end="\n\n")
print("Длина бутстраповского доверительного параметрического интервала: ")
print(f"l4 = {l4}")

Бутстраповский параметрический доверительный интервал: 
4.482 < θ < 6.159

Длина бутстраповского доверительного параметрического интервала: 
l4 = 1.677


### **ж) Сравнить все интервалы**

In [7]:
print("Асимптотический доверительный интервал медианы: ")
print(f"{np.round(start1,3)} < xm < {np.round(end1,3)}", end="\n\n")
print("Длина асимптотического доверительного интервала медианы: ")
print(f"l1 = {l1}")

Асимптотический доверительный интервал медианы: 
1.141 < xm < 1.217

Длина асимптотического доверительного интервала медианы: 
l1 = 0.076


In [8]:

print("Асимптотический доверительный интервал: ")
print(f"{np.round(start2,3)} < θ < {np.round(end2,3)}")
print("Бутстраповский непараметрический доверительный интервал: ")
print(f"{np.round(start3,3)} < θ < {np.round(end3,3)}")
print("Бутстраповский параметрический доверительный интервал: ")
print(f"{np.round(start4,3)} < θ < {np.round(end4,3)}", end="\n\n")

print("Длина асимптотического доверительного интервала: ")
print(f"l2 = {l2}")
print("Длина бутстраповского непараметрического доверительного интервала: ")
print(f"l3 = {l3}")
print("Длина бутстраповского доверительного параметрического интервала: ")
print(f"l4 = {l4}", end="\n\n")

ordered = np.array([
    [l2, "Асимптотический", f"{np.round(start2,3)} < θ < {np.round(end2,3)}"],
    [l3, "Бутстраповский непараметрический", f"{np.round(start3,3)} < θ < {np.round(end3,3)}"],
    [l4, "Бутстраповский параметрический", f"{np.round(start4,3)} < θ < {np.round(end4,3)}"],
                ])
winner = ordered[np.argsort(ordered[:,0])[0]]
print("Победитель: ")
print(f"{winner[1]} доверительный интервал:\nl = {winner[0]}")
print(winner[2])

Асимптотический доверительный интервал: 
4.379 < θ < 6.027
Бутстраповский непараметрический доверительный интервал: 
4.427 < θ < 6.224
Бутстраповский параметрический доверительный интервал: 
4.482 < θ < 6.159

Длина асимптотического доверительного интервала: 
l2 = 1.648
Длина бутстраповского непараметрического доверительного интервала: 
l3 = 1.797
Длина бутстраповского доверительного параметрического интервала: 
l4 = 1.677

Победитель: 
Асимптотический доверительный интервал:
l = 1.648
4.379 < θ < 6.027
